# 🔬 SemEval 2024 Task 8 — Subtask B | STAGGERED PARALLEL DRIVER
**Fix: Pre-download models to avoid parallel download hang**

---
### 🛠️ Verification Checklist
1. **Check Dataset**: Ensure `parallel_train.py`, `evaluate.py`, and JSONL files are attached.
2. **Check Accelerator**: Must be **GPU T4 x2**.
3. **Internet**: Must be **ON** (for first-time model download).
4. **Run All**: Models download first, then RoBERTa + ELECTRA train in parallel.

In [ ]:
!pip install -q transformers==4.39.3 accelerate sentencepiece huggingface_hub ujson

In [ ]:
import os, glob, shutil, subprocess, threading, sys, torch, time
from concurrent.futures import ThreadPoolExecutor

# 1. Copy scripts to working directory
scripts = ['parallel_train.py', 'evaluate.py']
for s in scripts:
    found = glob.glob(f'/kaggle/input/**/{s}', recursive=True)
    if found:
        shutil.copy(found[0], f'/kaggle/working/{s}')
        print(f"[OK] Copied {s} to /kaggle/working")
    else:
        print(f"[!] WARNING: {s} not found in input datasets!")

print(f"\nGPUs available: {torch.cuda.device_count()}")

## 📥 Phase 0: Pre-download Models (Sequential)
Downloads both models **one at a time** to `/kaggle/working/models/`.  
This avoids the hang caused by two GPU workers fighting over HuggingFace file locks.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

MODELS_TO_DOWNLOAD = {
    'roberta-base': 'roberta-base',
    'electra-base-discriminator': 'google/electra-base-discriminator',
}

MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)

for folder_name, model_id in MODELS_TO_DOWNLOAD.items():
    save_path = os.path.join(MODELS_DIR, folder_name)
    
    if os.path.exists(os.path.join(save_path, 'config.json')):
        print(f"[SKIP] {model_id} already cached at {save_path}")
        continue
    
    print(f"\n[DOWNLOADING] {model_id} ...")
    
    print(f"  Tokenizer...", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.save_pretrained(save_path)
    
    print(f"  Model weights...", flush=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=6)
    model.save_pretrained(save_path)
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"  [OK] Saved to {save_path}")

print("\n" + "="*60)
print("All models pre-downloaded! Training will use local files.")
print("="*60)

## 🚀 Phase 1: Simultaneous Training (Staggered Start)
RoBERTa starts now. ELECTRA will wait 30 seconds.

In [ ]:
import os, glob, shutil, subprocess, sys, torch, time
from concurrent.futures import ThreadPoolExecutor

def run_gpu_worker(model_config):
    m_name, gpu_id, delay = model_config
    if delay > 0:
        print(f"[WAIT] {m_name.upper()} Waiting {delay}s for staggered start...\n", flush=True)
        time.sleep(delay)
        
    cmd = ['python', '-u', '/kaggle/working/parallel_train.py', f'--model={m_name}', f'--gpu={gpu_id}']
    
    # Directly inherit stdout/stderr so in-place terminal overwriting works natively!
    p = subprocess.Popen(cmd)
    exit_code = p.wait()
    return f"\n[FINISHED] {m_name.upper()} exited (code {exit_code})"

print("="*60)
print("STARTING STAGGERED PARALLEL TRAINING")
print("="*60 + "\n")

with ThreadPoolExecutor(max_workers=2) as executor:
    configs = [('roberta', 0, 0), ('electra', 1, 30)]
    futures = [executor.submit(run_gpu_worker, c) for c in configs]
    for f in futures:
        print(f.result())


## 📊 Phase 2: Ensemble Evaluation & Plotting

In [ ]:
print("\n" + "="*60)
print("LAUNCHING ENSEMBLE EVALUATION")
print("="*60)
!python /kaggle/working/evaluate.py --mode both --split both

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Final_Ensemble_Results', 'zip', '/kaggle/working/plots')
print("\n--- COMPLETE ---")
print("1. All plots and metrics zipped! Download 'Final_Ensemble_Results.zip'")
print("2. Check your HuggingFace Hub for the updated model and plots repository.")